Create a catalog on external location

In [0]:
%sql
create catalog ext_cat managed location 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2'

In [0]:
%sql
use catalog ext_cat

Create a volume in the external location

In [0]:
%sql
create volume ext_cat.default.managedExtVol;

In [0]:
%sql
show volumes

database,volume_name
default,managedextvol


Create a directory

In [0]:
%fs
mkdirs /Volumes/ext_cat/default/managedextvol/emp

res5: Boolean = true

Upload a file to the volume

In [0]:
%fs
ls  /Volumes/ext_cat/default/managedextvol/emp

path,name,size,modificationTime
dbfs:/Volumes/ext_cat/default/managedextvol/emp/employees.csv,employees.csv,3778,1788243786000


Ways to read the data

In [0]:
%sql
select * from csv.`/Volumes/ext_cat/default/managedextvol/emp/`

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10
EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
198,Donald,OConnell,DOCONNEL,650.507.9833,21-JUN-07,SH_CLERK,2600,-,124,50
199,Douglas,Grant,DGRANT,650.507.9844,13-JAN-08,SH_CLERK,2600,-,124,50
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-SEP-03,AD_ASST,4400,-,101,10
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-FEB-04,MK_MAN,13000,-,100,20
202,Pat,Fay,PFAY,603.123.6666,17-AUG-05,MK_REP,6000,-,201,20
203,Susan,Mavris,SMAVRIS,515.123.7777,07-JUN-02,HR_REP,6500,-,101,40
204,Hermann,Baer,HBAER,515.123.8888,07-JUN-02,PR_REP,10000,-,101,70
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-JUN-02,AC_MGR,12008,-,101,110
206,William,Gietz,WGIETZ,515.123.8181,07-JUN-02,AC_ACCOUNT,8300,-,205,110


In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/ext_cat/default/managedextvol/emp/employees.csv',
  format => 'csv',
  header => 'true'
);

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,_rescued_data
198,Donald,OConnell,DOCONNEL,650.507.9833,21-JUN-07,SH_CLERK,2600,-,124,50,null
199,Douglas,Grant,DGRANT,650.507.9844,13-JAN-08,SH_CLERK,2600,-,124,50,null
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-SEP-03,AD_ASST,4400,-,101,10,null
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-FEB-04,MK_MAN,13000,-,100,20,null
202,Pat,Fay,PFAY,603.123.6666,17-AUG-05,MK_REP,6000,-,201,20,null
203,Susan,Mavris,SMAVRIS,515.123.7777,07-JUN-02,HR_REP,6500,-,101,40,null
204,Hermann,Baer,HBAER,515.123.8888,07-JUN-02,PR_REP,10000,-,101,70,null
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-JUN-02,AC_MGR,12008,-,101,110,null
206,William,Gietz,WGIETZ,515.123.8181,07-JUN-02,AC_ACCOUNT,8300,-,205,110,null
100,Steven,King,SKING,515.123.4567,17-JUN-03,AD_PRES,24000,-,-,90,null


In [0]:
%sql
drop volume ext_cat.default.managedExtVol;

In [0]:
%sql
create volume ext_cat.default.managedExtVol;

When we drop a volume, even though its external, and the files are still present in external storage ,  if we re create it with same name, the new volume will be empty. This is because databricks unity catalog creates a new folder in external storage for the newly created volume

In [0]:
%fs
ls /Volumes/ext_cat/default/managedextvol

In [0]:
%sql
select * from csv.`/Volumes/ext_cat/default/managedextvol/emp/`

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6170675731086251>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from csv.`/Volumes/ext_cat/default/managedextvol/emp/`\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     raise Exception(
    127         "Can